# Asistente RAG sobre el corpus de tweets — COVID Colombia 2020

Adaptación del cuaderno `rag_columnas_pycon2026.ipynb` al corpus del proyecto.

**Diferencias con el original:**

| Aspecto | Original (PyCon) | Este cuaderno |
|---|---|---|
| Corpus | Excel de columnas subido a mano en Colab | **Lectura automática** de `resultadosCOMB/corpus_cleaned.parquet` (151.424 tweets) |
| Fragmentación | Trozos de 400 caracteres con solape | **1 tweet = 1 fragmento** (los tweets ya son cortos) |
| Embeddings | MiniLM, calculados en vivo | **Reutiliza `tweet_embeddings.npy`** del pipeline (mpnet, ya calculados) — solo se encodea la pregunta |
| Generación | Groq o Mistral | **Solo Mistral (API)** o **modelo local en tu GPU**, conmutables con una variable |

El flujo RAG es el mismo: (1) buscar los tweets más cercanos a la pregunta por similitud de embeddings, (2) entregarlos como contexto a un modelo de lenguaje que redacta la respuesta citando cada afirmación con el número del fragmento.


## 0 · Configuración

`GENERADOR` controla quién redacta la respuesta final:

- `'api'` → Mistral por API (necesita `MISTRAL_API_KEY`, se pide más abajo).
- `'local'` → modelo abierto corriendo en tu GPU (4-bit, apto para 4–6 GB de VRAM).

Puedes cambiarlo aquí o pasarlo por llamada: `responder_rag(pregunta, generador='local')`.


In [1]:
# ============================================================
# CELL 0 — CONFIG
# ============================================================
from pathlib import Path

# --- Datos (lectura automática, sin subir nada) ---
DATA_PROCESSED  = Path(r'C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosCOMB')
RUTA_CORPUS_PQ  = DATA_PROCESSED / 'corpus_cleaned.parquet'
RUTA_CORPUS_XL  = DATA_PROCESSED / 'corpus_cleaned.xlsx'      # respaldo si no hay parquet
RUTA_EMBEDDINGS = DATA_PROCESSED / 'tweet_embeddings.npy'     # reutilizados del pipeline (NB03)

# --- Recuperación ---
MODELO_EMB = 'paraphrase-multilingual-mpnet-base-v2'  # el MISMO del pipeline: los .npy son de este modelo
K_DEFECTO  = 6                                        # fragmentos a recuperar por pregunta

# --- Generación ---
GENERADOR      = 'api'                     # 'api' (Mistral) | 'local' (tu GPU)
MODELO_MISTRAL = 'mistral-small-latest'    # alternativa: 'mistral-large-latest'
MODELO_LOCAL   = 'Qwen/Qwen2.5-7B-Instruct'  # 4-bit ~4.3 GB VRAM (cabe con ~4.9 GB libres)
# Alternativas segun VRAM libre:
#   ~4.5+ GB : 'Qwen/Qwen2.5-7B-Instruct' | 'mistralai/Mistral-7B-Instruct-v0.3'
#   ~2.5 GB  : 'Qwen/Qwen2.5-3B-Instruct'
#   ~1.5 GB  : 'Qwen/Qwen2.5-1.5B-Instruct'
TEMPERATURA    = 0.2

print('[CONFIG] OK')
print(f'  Corpus     : {RUTA_CORPUS_PQ.name}')
print(f'  Embeddings : {RUTA_EMBEDDINGS.name}')
print(f'  Generador  : {GENERADOR}')


[CONFIG] OK
  Corpus     : corpus_cleaned.parquet
  Embeddings : tweet_embeddings.npy
  Generador  : api


## 1 · Instalación

Se instala una sola vez. `faiss-cpu` es el índice vectorial; `openai` es el cliente
(la API de Mistral es compatible con ese formato). Las tres últimas solo hacen falta
para el modelo local.


In [2]:
# ============================================================
# CELL 1 — DEPENDENCIAS
# Descomenta la primera vez:
# ============================================================
# %pip install -q sentence-transformers faiss-cpu openai pandas openpyxl pyarrow
# %pip install -q transformers accelerate bitsandbytes   # solo para GENERADOR='local'
# (tras instalar, reinicia el kernel: Kernel > Restart)

import importlib.util
print('Para la recuperacion y la API:')
for paquete in ['sentence_transformers', 'faiss', 'openai', 'pandas']:
    ok = importlib.util.find_spec(paquete) is not None
    print(f"  {'OK   ' if ok else 'FALTA'} {paquete}")
print('Solo para el modelo local (GENERADOR=\'local\'):')
for paquete in ['transformers', 'accelerate', 'bitsandbytes']:
    ok = importlib.util.find_spec(paquete) is not None
    print(f"  {'OK   ' if ok else 'FALTA'} {paquete}")


Para la recuperacion y la API:
  OK    sentence_transformers
  FALTA faiss
  OK    openai
  OK    pandas
Solo para el modelo local (GENERADOR='local'):
  OK    transformers
  OK    accelerate
  OK    bitsandbytes


## 2 · Clave de API de Mistral

Solo se necesita si `GENERADOR = 'api'`. Se obtiene gratis en `console.mistral.ai`.
`getpass` no muestra la clave en pantalla; también puedes definir la variable de
entorno `MISTRAL_API_KEY` en Windows y esta celda la tomará sola.


In [3]:
# ============================================================
# CELL 2 — CLAVE MISTRAL
# ============================================================
import os
from getpass import getpass

if GENERADOR == 'api' and not os.environ.get('MISTRAL_API_KEY'):
    clave = getpass('Clave de API de Mistral (Enter para omitir): ')
    if clave:
        os.environ['MISTRAL_API_KEY'] = clave

print('Mistral configurada:', bool(os.environ.get('MISTRAL_API_KEY')))


Mistral configurada: True


## 3 · Carga automática de datos

Se lee directamente el corpus limpio del pipeline (el mismo de todas las réplicas).
No hay fragmentación: cada tweet (~30 palabras) ya es una unidad de sentido acotada,
igual que en el pipeline donde 1 tweet = 1 chunk. Los metadatos (autor, tipo de
cuenta, fecha, id) se conservan para construir las citas.


In [4]:
# ============================================================
# CELL 3 — CARGA AUTOMATICA DEL CORPUS
# ============================================================
import pandas as pd

if RUTA_CORPUS_PQ.exists():
    corpus = pd.read_parquet(RUTA_CORPUS_PQ)
    print(f'[CARGA] {RUTA_CORPUS_PQ.name}')
elif RUTA_CORPUS_XL.exists():
    corpus = pd.read_excel(RUTA_CORPUS_XL)
    print(f'[CARGA] {RUTA_CORPUS_XL.name}')
else:
    raise FileNotFoundError('No se encontró corpus_cleaned. Corre antes el notebook 01 del pipeline.')

# Cada tweet = 1 fragmento. Metadatos para las citas.
frag_df = corpus[['id_doc', 'Author_Normalized', 'Entidad', 'Fecha', 'Texto_limpio']].copy()
frag_df = frag_df.rename(columns={'Texto_limpio': 'fragmento'})
frag_df['Fecha'] = pd.to_datetime(frag_df['Fecha'], errors='coerce').dt.strftime('%Y-%m-%d')
frag_df = frag_df.dropna(subset=['fragmento']).reset_index(drop=True)

print(f'Fragmentos (tweets) : {len(frag_df):,}')
frag_df.head(3)


[CARGA] corpus_cleaned.parquet
Fragmentos (tweets) : 151,424


,id_doc,Author_Normalized,Entidad,Fecha,fragmento
0,1,@treporta,"Noticias locales, nacionales o globales",2020-06-03,VDEO Reos que no están contagiados con COVID19...
1,2,@annytak21,Sin descripción,2020-06-03,ya se enteraron de la muerte supestamente por ...
2,3,@ferumapress,"Noticias locales, nacionales o globales",2020-06-03,El calvario de una familia en Cali por su mamá...


## 4 · Embeddings — reutilización del pipeline

El pipeline ya calculó los embeddings de los 151.424 tweets con
`paraphrase-multilingual-mpnet-base-v2` (NB03). Aquí se cargan y se **normalizan**
(norma 1), de modo que la similitud del coseno equivale al producto punto — lo que
esperan tanto la búsqueda con NumPy como el índice FAISS `IndexFlatIP`.

El modelo solo se usa en vivo para encodear **la pregunta** (milisegundos).
Si el `.npy` no existiera, la celda recalcula todo desde cero.


In [5]:
# ============================================================
# CELL 4 — EMBEDDINGS (cargar y normalizar; calcular solo si faltan)
# ============================================================
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
modelo_emb = SentenceTransformer(MODELO_EMB, device=device)
print(f'Modelo de embeddings : {MODELO_EMB}  (device: {device})')

if RUTA_EMBEDDINGS.exists():
    emb = np.load(RUTA_EMBEDDINGS).astype('float32')
    if len(emb) != len(frag_df):
        raise ValueError(f'Embeddings ({len(emb)}) no alinean con el corpus ({len(frag_df)}). '
                         'Regenera con RELOAD=True en el NB03 o borra el .npy.')
    print(f'[REUTILIZADO] {RUTA_EMBEDDINGS.name}  {emb.shape}')
else:
    print('No hay embeddings previos; calculando (puede tardar)...')
    emb = modelo_emb.encode(frag_df['fragmento'].tolist(), batch_size=128,
                            show_progress_bar=True).astype('float32')
    np.save(RUTA_EMBEDDINGS, emb)

# Normalizar a norma 1 -> coseno == producto punto
normas = np.linalg.norm(emb, axis=1, keepdims=True)
normas[normas == 0] = 1.0
emb = emb / normas
print(f'Matriz normalizada   : {emb.shape}')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo de embeddings : paraphrase-multilingual-mpnet-base-v2  (device: cuda)
[REUTILIZADO] tweet_embeddings.npy  (151424, 768)
Matriz normalizada   : (151424, 768)


## 5 · Índice y búsqueda

Dos implementaciones equivalentes, como en el original: NumPy (la operación a la
vista: un producto de matrices) y FAISS (`IndexFlatIP`, escala mejor). Con 151 mil
vectores ambas responden en fracciones de segundo.


In [6]:
# ============================================================
# CELL 5 — BUSQUEDA (NumPy + FAISS)
# ============================================================
def _encajar(consulta):
    v = modelo_emb.encode([consulta])
    v = np.asarray(v, dtype='float32')
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    return v

def buscar_numpy(consulta, k=K_DEFECTO):
    v = _encajar(consulta)[0]
    puntajes = emb @ v
    idx = np.argsort(-puntajes)[:k]
    resultados = []
    for pos in idx:
        fila = frag_df.iloc[int(pos)].to_dict()
        fila['score'] = float(puntajes[int(pos)])
        resultados.append(fila)
    return resultados

try:
    import faiss
    indice = faiss.IndexFlatIP(emb.shape[1])
    indice.add(emb)
    def buscar(consulta, k=K_DEFECTO):
        v = _encajar(consulta)
        puntajes, idx = indice.search(v, k)
        resultados = []
        for pos, score in zip(idx[0], puntajes[0]):
            fila = frag_df.iloc[int(pos)].to_dict()
            fila['score'] = float(score)
            resultados.append(fila)
        return resultados
    print(f'Indice FAISS con {indice.ntotal:,} vectores.')
except ImportError:
    buscar = buscar_numpy
    print('faiss no instalado: se usa la búsqueda NumPy (mismo resultado).')

for r in buscar('escasez de camas UCI en los hospitales', k=3):
    print(round(r['score'], 3), '|', r['fragmento'][:100])


faiss no instalado: se usa la búsqueda NumPy (mismo resultado).
0.757 | En la previa de los anuncios de restricciones de movilidad del Minsal, el Comité Científico Covid19 
0.756 | Covid 19 SITUACIN HOSPITALARIA GRAVE No hay camas UCI disponibles. Médicos buscan colocar pacientes 
0.753 | Por el Covid19 hospital en San Luis Potosí se queda sin camas para atender pacientes. Ya no hay luga


## 6 · Recuperación con su fuente

Antes de involucrar al modelo de lenguaje, conviene ver qué recupera la búsqueda.
Cada fragmento se imprime con su metadato: autor, tipo de cuenta, fecha e id del tweet.


In [7]:
# ============================================================
# CELL 6 — MOSTRAR FRAGMENTOS RECUPERADOS
# ============================================================
def mostrar_fragmentos(resultados):
    for n, r in enumerate(resultados, start=1):
        print(f"[{n}] {r['Author_Normalized']}  ({r['Entidad']}, {r['Fecha']})")
        print(f"     id_doc: {r['id_doc']}  |  similitud: {round(r['score'], 3)}")
        print(f"     {r['fragmento']}")
        print()

PREGUNTA = '¿Qué se dice sobre la vacuna contra el COVID?'
mostrar_fragmentos(buscar(PREGUNTA, k=5))


[1] @navarra105  (Autoridad local de salud o gobierno, 2020-08-09)
     id_doc: 74898  |  similitud: 0.9
     Así es la vacuna contra el Covid19

[2] @opinometroo  (Noticias locales, nacionales o globales, 2020-06-04)
     id_doc: 137172  |  similitud: 0.887
     ¿Crees que se encuentre vacuna para COVID19 ?

[3] @plosckar  (Autoridad local de salud o gobierno, 2020-08-18)
     id_doc: 101152  |  similitud: 0.878
     Un poco menos de desinformación sobre la vacuna del COVID19

[4] @laopini40286861  (Noticias locales, nacionales o globales, 2020-05-28)
     id_doc: 122782  |  similitud: 0.863
     Y yo abono a la pregunta: Ya tenemos la vacuna y el tratamiento contra el Covid19?

[5] @el_galacticoz20  (Noticias locales, nacionales o globales, 2020-05-21)
     id_doc: 34164  |  similitud: 0.85
     COVID19 una vez que venga esa vacuna



## 7 · Generación — dos vías conmutables

La respuesta final la puede redactar:

**a) Mistral por API** — la API de Mistral acepta el formato de la API de OpenAI,
así que basta el cliente `openai` apuntando a `api.mistral.ai`.

**b) Un modelo local en tu GPU** — cuantizado a 4 bits y forzado 100 % a la GPU
(si no cabe, falla con OOM en lugar de repartirse a CPU/disco sin avisar).

| VRAM libre | Modelo local | VRAM aprox (4-bit) |
|---|---|---|
| ~4,5+ GB | `Qwen/Qwen2.5-7B-Instruct` (por defecto) | ~4,3 GB |
| ~4,5+ GB | `mistralai/Mistral-7B-Instruct-v0.3` | ~4,3 GB |
| ~2,5 GB | `Qwen/Qwen2.5-3B-Instruct` | ~2,2 GB |
| ~1,5 GB | `Qwen/Qwen2.5-1.5B-Instruct` | ~1,2 GB |

La celda de diagnóstico mide tu VRAM libre (tras mover los embeddings a CPU) y
avisa si el modelo elegido no cabe. Con 6 GB totales y ~4,9 libres, el 7B entra justo:
cierra otras apps que usen GPU (navegador con aceleración de hardware, juegos).


In [8]:
# ============================================================
# CELL 7 — DIAGNOSTICO DE GPU
# ============================================================
import torch

if torch.cuda.is_available():
    nombre = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU        : {nombre}')
    print(f'VRAM total : {vram_gb:.1f} GB')
    if vram_gb >= 7.5:
        print('-> Cabe Mistral-7B-Instruct-v0.3 en 4-bit. Puedes poner:')
        print("   MODELO_LOCAL = 'mistralai/Mistral-7B-Instruct-v0.3'")
    elif vram_gb >= 3.5:
        print(f'-> Adecuado para modelos ~3B en 4-bit. MODELO_LOCAL actual: {MODELO_LOCAL}')
    else:
        print('-> VRAM muy limitada: usa GENERADOR = "api".')
else:
    print('No se detecta GPU CUDA. Usa GENERADOR = "api" (el local iría por CPU, muy lento).')


GPU        : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM total : 6.0 GB
-> Adecuado para modelos ~3B en 4-bit. MODELO_LOCAL actual: Qwen/Qwen2.5-7B-Instruct


In [9]:
# ============================================================
# CELL 8 — GENERACION VIA API MISTRAL
# ============================================================
from openai import OpenAI

def generar_api(prompt, temperatura=TEMPERATURA):
    clave = os.environ.get('MISTRAL_API_KEY')
    if not clave:
        raise RuntimeError('Falta MISTRAL_API_KEY (corre la celda 2).')
    cliente = OpenAI(base_url='https://api.mistral.ai/v1', api_key=clave)
    respuesta = cliente.chat.completions.create(
        model=MODELO_MISTRAL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperatura,
    )
    return respuesta.choices[0].message.content


In [10]:
# ============================================================
# CELL 9 — GENERACION LOCAL EN GPU (carga perezosa, 4-bit)
# Requiere: pip install transformers accelerate bitsandbytes
# El modelo se carga la primera vez que se usa y queda en memoria.
# ============================================================
import importlib.util

_pipe_local = None

def _revisar_deps_local():
    faltan = [p for p in ['transformers', 'accelerate'] if importlib.util.find_spec(p) is None]
    if faltan:
        raise RuntimeError(
            f"Faltan dependencias para el modelo local: {', '.join(faltan)}. "
            "Instala con:  %pip install transformers accelerate bitsandbytes  "
            "y reinicia el kernel."
        )
    return importlib.util.find_spec('bitsandbytes') is not None

def _liberar_vram():
    """Mueve el modelo de embeddings a CPU y limpia cache CUDA.
    El mpnet ocupa ~1 GB de VRAM y para encodear UNA pregunta la CPU sobra.
    Ese GB suele ser la diferencia entre que el LLM 4-bit quepa o no."""
    try:
        modelo_emb.to('cpu')
        print('  Modelo de embeddings movido a CPU (las consultas siguen funcionando).')
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        libre, total = torch.cuda.mem_get_info()
        print(f'  VRAM libre: {libre/1024**3:.1f} / {total/1024**3:.1f} GB')
        return libre / 1024**3
    return 0.0

def _cargar_local():
    global _pipe_local
    if _pipe_local is not None:
        return _pipe_local

    hay_bnb = _revisar_deps_local()
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    print(f'Cargando modelo local {MODELO_LOCAL} (primera vez tarda unos minutos)...')

    vram_libre = _liberar_vram()
    NECESITA = 4.5 if '7B' in MODELO_LOCAL else (2.5 if '3B' in MODELO_LOCAL else 1.5)
    if vram_libre and vram_libre < NECESITA:
        print(f'  [AVISO] {vram_libre:.1f} GB libres y {MODELO_LOCAL} necesita ~{NECESITA} GB en 4-bit.')
        print('          Cierra apps que usen GPU (navegador con aceleracion, juegos) o baja de tamano:')
        print("          7B -> 'Qwen/Qwen2.5-3B-Instruct' -> 'Qwen/Qwen2.5-1.5B-Instruct'")

    modelo = None
    if hay_bnb:
        try:
            from transformers import BitsAndBytesConfig
            bnb = BitsAndBytesConfig(load_in_4bit=True,
                                     bnb_4bit_compute_dtype=torch.float16,
                                     bnb_4bit_quant_type='nf4',
                                     bnb_4bit_use_double_quant=True)  # ahorra ~0.4 GB extra
            # device_map={'': 0} fuerza TODO el modelo a la GPU: si no cabe,
            # falla con OOM en vez de repartirse en CPU/disco sin avisar.
            modelo = AutoModelForCausalLM.from_pretrained(
                MODELO_LOCAL, quantization_config=bnb, device_map={'': 0})
            print('  Cuantizado 4-bit, 100% en GPU.')
            if torch.cuda.is_available():
                usado = torch.cuda.memory_allocated() / 1024**3
                print(f'  VRAM ocupada por el modelo: {usado:.1f} GB')
        except Exception as e:
            print(f'  [AVISO] 4-bit fallo ({type(e).__name__}: {e}); probando float16...')
    else:
        print('  [AVISO] bitsandbytes no instalado — sin 4-bit. Un 3B en float16 ocupa ~6.2 GB:')
        print('          en una GPU de 4-6 GB puede no caber. Instala bitsandbytes:')
        print('          %pip install bitsandbytes   (y reinicia el kernel)')

    if modelo is None:
        # Fallback float16 con device_map (accelerate reparte GPU/CPU si no cabe)
        modelo = AutoModelForCausalLM.from_pretrained(
            MODELO_LOCAL, dtype=torch.float16, device_map='auto')
        print('  Cargado en float16 con device_map=auto (partes en CPU si la VRAM no alcanza).')
        print('  [AVISO] Si ves "offloaded to the disk", la generacion sera MUY lenta:')
        print('          mejor reinicia el kernel y usa un modelo mas pequeno en 4-bit.')

    tok = AutoTokenizer.from_pretrained(MODELO_LOCAL)
    _pipe_local = pipeline('text-generation', model=modelo, tokenizer=tok)
    return _pipe_local

def generar_local(prompt, temperatura=TEMPERATURA, max_tokens=600):
    pipe = _cargar_local()
    from transformers import GenerationConfig
    # Un solo GenerationConfig evita los avisos de 'generation_config junto a
    # argumentos sueltos' y de 'max_new_tokens vs max_length'.
    gen_cfg = GenerationConfig(
        max_new_tokens=max_tokens,
        do_sample=temperatura > 0,
        temperature=max(temperatura, 0.01),
        pad_token_id=pipe.tokenizer.eos_token_id,
    )
    mensajes = [
        {'role': 'system', 'content': (
            'Responde SIEMPRE como lista de items: cada linea empieza con "- ". '
            'Cada afirmacion termina con la cita del fragmento entre corchetes, por ejemplo [1]. '
            'No escribas parrafos corridos. Responde solo con la informacion del contexto.')},
        {'role': 'user', 'content': prompt},
    ]
    salida = pipe(mensajes, generation_config=gen_cfg, return_full_text=False)
    return salida[0]['generated_text']


In [11]:
# ============================================================
# CELL 10 — CONMUTADOR
# ============================================================
def generar(prompt, generador=None, temperatura=TEMPERATURA):
    g = generador or GENERADOR
    if g == 'api':
        return generar_api(prompt, temperatura)
    if g == 'local':
        return generar_local(prompt, temperatura)
    raise ValueError(f"Generador desconocido: {g} (usa 'api' o 'local')")

print(f'Generador activo por defecto: {GENERADOR}')


Generador activo por defecto: api


## 8 · Función RAG completa

Recupera los `k` tweets más cercanos, los numera dentro del prompt y pide al modelo
responder solo con esa información, citando cada afirmación con el número del
fragmento entre corchetes. Después imprime los fragmentos citados con su metadato,
de modo que cada cita se puede rastrear hasta el tweet de origen.


In [12]:
# ============================================================
# CELL 11 — PROMPT Y FUNCION RAG
# ============================================================
def construir_prompt(pregunta, fragmentos):
    contexto = ''
    for n, r in enumerate(fragmentos, start=1):
        contexto += (
            f"[{n}] @{r['Author_Normalized']} ({r['Entidad']}, {r['Fecha']}, id {r['id_doc']})\n"
            f"{r['fragmento']}\n\n"
        )
    return (
        'Eres un asistente que responde unicamente con la informacion del contexto.\n'
        'Si el contexto no contiene la respuesta, di que no encuentras la informacion, '
        'pero resume la informacion relacionada.\n'
        'Usa items.\n'
        'Cita cada afirmacion con el numero del fragmento entre corchetes, por ejemplo [1].\n\n'
        f'Contexto:\n{contexto}'
        f'Pregunta: {pregunta}\n'
        'Respuesta:'
    )

def responder_rag(pregunta, k=K_DEFECTO, generador=None):
    fragmentos = buscar(pregunta, k=k)
    prompt = construir_prompt(pregunta, fragmentos)
    respuesta = generar(prompt, generador=generador)
    print(respuesta)
    print()
    print('--- Fragmentos utilizados ' + '-' * 34)
    mostrar_fragmentos(fragmentos)
    return respuesta, fragmentos


In [13]:
# ============================================================
# CELL 12 — EJEMPLO DE USO
# ============================================================
_ = responder_rag('¿Qué barreras se mencionan para acceder a atención en salud durante la pandemia?', k=6)


- Restricción en el acceso a insumos médicos durante la pandemia [6].
- Falta de garantías en equipos de protección personal (EPP) para profesionales de la salud [3].
- Cobertura insuficiente por parte de las ARL (Administradoras de Riesgos Laborales) en casos de infección por COVID-19 como enfermedad laboral [3].

--- Fragmentos utilizados ----------------------------------
[1] @cirofernandezn  (Autoridad local de salud o gobierno, 2020-05-08)
     id_doc: 148463  |  similitud: 0.815
     Si bien es cierto la pandemia se ha manejado de buena manera, no podemos dejar sin elementos de protección a nuestros médicos y sin recursos a nuestros hospitales, sobre todo en la zona rural. COVID19 Santander Barrancabermeja

[2] @paterninamd  (Autoridad local de salud o gobierno, 2020-05-10)
     id_doc: 2958  |  similitud: 0.798
     Es claro que se necesitan cambios en los sistemas de salud, mejor preparación y más agilidad, y no solo para pandemias poco frecuentes. Pero la reestructuración de l

## 9 · Comparación API vs. modelo local

La misma pregunta, con los mismos fragmentos recuperados, pasa por las dos vías.
Si una no está disponible (sin clave, sin GPU), el bloque lo informa y sigue con la otra.


In [20]:
# ============================================================
# CELL 13 — COMPARACION API vs LOCAL
# ============================================================
PREGUNTA_COMPARACION = '¿Qué se dice sobre el periodismo?'
fragmentos = buscar(PREGUNTA_COMPARACION, k=6)
prompt = construir_prompt(PREGUNTA_COMPARACION, fragmentos)

for g in ['api', 'local']:
    print('=' * 60)
    print('Generador:', g)
    print('=' * 60)
    try:
        print(generar(prompt, generador=g))
    except Exception as e:
        print(f'[{g} no disponible] {e}')
    print()

print('--- Fragmentos utilizados ' + '-' * 34)
mostrar_fragmentos(fragmentos)


Generador: api
- Se critica la falta de ética y seriedad en algunos periodistas, acusándolos de generar desinformación y crear pánico [6].
- Se cuestiona la credibilidad de ciertos medios y periodistas, señalando que no verifican bien la información y pueden ser tendenciosos [3].
- Se menciona que algunos periodistas no deberían publicar noticias sin ser expertos en el tema o funcionarios oficiales [5].
- Se destaca que hay periodistas que comparten información con capturas como respaldo, buscando dar veracidad a sus notas [4].
- Se observa que algunos presentadores de noticias tienen viviendas lujosas, lo que genera comentarios sobre el aspecto económico del periodismo [1].
- Se menciona una unión de medios para enfrentar el Covid-19 en el Huila, mostrando un esfuerzo colectivo en el periodismo local [2].

Generador: local
- Se critica la falta de ética en algunos medios [6]
- Se menciona que algunos periodistas son serios [6]
- Se sugiere que los periodistas deben informar correctame